In [8]:
# 安裝必要套件
!pip install pycaret optuna


In [9]:
import pandas as pd

# 載入 Titanic 數據集
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
titanic = pd.read_csv(url)

# 簡單數據清理
titanic['Age'].fillna(titanic['Age'].median(), inplace=True)
titanic['Embarked'].fillna('S', inplace=True)

# 刪除不必要欄位
titanic.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)


In [12]:
clf_setup = setup(
    data=titanic,
    target='Survived',
    train_size=0.8,
    session_id=42,
    normalize=True,
    categorical_features=['Sex', 'Embarked'],
    polynomial_features=True,
    remove_multicollinearity=True,
    log_experiment=False,  # 關閉實驗記錄
    experiment_name='titanic_optuna_demo'
)


,Description,Value
0,Session id,42
1,Target,Survived
2,Target type,Binary
3,Original data shape,"(891, 8)"
4,Transformed data shape,"(891, 41)"
5,Transformed train set shape,"(712, 41)"
6,Transformed test set shape,"(179, 41)"
7,Numeric features,5
8,Categorical features,2
9,Preprocess,True


In [13]:
# 比較所有模型，選出最佳模型
best_model = compare_models()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lr,Logistic Regression,0.8245,0.8530,0.6702,0.8411,0.7386,0.6113,0.6244,0.6450
ridge,Ridge Classifier,0.8231,0.8542,0.6593,0.8493,0.7328,0.6065,0.6222,0.1030
gbc,Gradient Boosting Classifier,0.8217,0.8661,0.7075,0.8031,0.7463,0.6114,0.6186,0.4210
lda,Linear Discriminant Analysis,0.8216,0.8544,0.6519,0.8519,0.7275,0.6019,0.6191,0.1210
ada,Ada Boost Classifier,0.8147,0.8401,0.7295,0.7760,0.7478,0.6023,0.6069,0.3020
knn,K Neighbors Classifier,0.8049,0.8442,0.6960,0.7741,0.7296,0.5783,0.5830,0.1910
lightgbm,Light Gradient Boosting Machine,0.8034,0.8632,0.7185,0.7565,0.7324,0.5782,0.5830,0.7180
rf,Random Forest Classifier,0.7978,0.8614,0.7181,0.7461,0.7275,0.5677,0.5720,0.2890
xgboost,Extreme Gradient Boosting,0.7893,0.8512,0.7221,0.7262,0.7201,0.5520,0.5558,0.2040
et,Extra Trees Classifier,0.7837,0.8236,0.7110,0.7205,0.7125,0.5398,0.5429,0.2500


Processing:   0%|          | 0/65 [00:00<?, ?it/s]

In [15]:
import optuna
from pycaret.classification import *

# 定義目標函數
def objective(trial):
    # 定義超參數空間
    max_depth = trial.suggest_int('max_depth', 2, 10)
    n_estimators = trial.suggest_int('n_estimators', 50, 500)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)

    # 訓練模型
    model = create_model('xgboost',
                         max_depth=max_depth,
                         n_estimators=n_estimators,
                         learning_rate=learning_rate)

    # 提取模型評估結果
    results = pull()
    accuracy = results.loc[0, 'Accuracy']  # 提取準確率
    return accuracy

# 使用 Optuna 進行優化
study = optuna.create_study(direction='maximize')  # 最大化準確率
study.optimize(objective, n_trials=50)  # 試驗次數

print("最佳參數組合:", study.best_params)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7361,0.7963,0.5000,0.7368,0.5957,0.4103,0.4274
1,0.8472,0.8888,0.7500,0.8400,0.7925,0.6722,0.6749
2,0.8310,0.8596,0.7143,0.8333,0.7692,0.6371,0.6419
3,0.7465,0.7917,0.4074,0.8462,0.5500,0.4022,0.4543
4,0.8451,0.8792,0.7778,0.8077,0.7925,0.6689,0.6692
5,0.8873,0.9125,0.8519,0.8519,0.8519,0.7609,0.7609
6,0.9155,0.8998,0.8148,0.9565,0.8800,0.8154,0.8217
7,0.8310,0.8923,0.8148,0.7586,0.7857,0.6465,0.6476
8,0.8028,0.8565,0.5926,0.8421,0.6957,0.5562,0.5750


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7500,0.7825,0.5714,0.7273,0.6400,0.4527,0.4604
1,0.8472,0.8872,0.7500,0.8400,0.7925,0.6722,0.6749
2,0.8028,0.8559,0.7143,0.7692,0.7407,0.5820,0.5830
3,0.7324,0.7479,0.4444,0.7500,0.5581,0.3837,0.4108
4,0.8451,0.8746,0.7407,0.8333,0.7843,0.6641,0.6669
5,0.8310,0.9327,0.8519,0.7419,0.7931,0.6514,0.6558
6,0.9014,0.9209,0.7778,0.9545,0.8571,0.7831,0.7926
7,0.8169,0.8847,0.7778,0.7500,0.7636,0.6143,0.6146
8,0.8310,0.8594,0.6667,0.8571,0.7500,0.6253,0.6366


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7222,0.7711,0.5000,0.7000,0.5833,0.3836,0.3958
1,0.8472,0.8864,0.7500,0.8400,0.7925,0.6722,0.6749
2,0.7746,0.8754,0.6786,0.7308,0.7037,0.5223,0.5232
3,0.6479,0.7315,0.4444,0.5455,0.4898,0.2252,0.2280
4,0.8028,0.8636,0.7407,0.7407,0.7407,0.5816,0.5816
5,0.7887,0.9217,0.8519,0.6765,0.7541,0.5731,0.5849
6,0.8732,0.9015,0.7778,0.8750,0.8235,0.7252,0.7282
7,0.8028,0.8796,0.8148,0.7097,0.7586,0.5933,0.5973
8,0.8028,0.8468,0.6296,0.8095,0.7083,0.5629,0.5730


...

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7361,0.7833,0.5714,0.6957,0.6275,0.4262,0.4311
1,0.8472,0.8864,0.7500,0.8400,0.7925,0.6722,0.6749
2,0.7324,0.8663,0.6786,0.6552,0.6667,0.4433,0.4434
3,0.6620,0.7155,0.5185,0.5600,0.5385,0.2724,0.2729
4,0.8028,0.8662,0.7407,0.7407,0.7407,0.5816,0.5816
5,0.8169,0.9125,0.8519,0.7188,0.7797,0.6249,0.6316
6,0.8873,0.9099,0.8148,0.8800,0.8462,0.7575,0.7589
7,0.7746,0.8678,0.7778,0.6774,0.7241,0.5352,0.5388
8,0.7887,0.8544,0.6296,0.7727,0.6939,0.5351,0.5417


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7361,0.7776,0.5714,0.6957,0.6275,0.4262,0.4311
1,0.8333,0.8807,0.8214,0.7667,0.7931,0.6538,0.6549
2,0.8028,0.8821,0.7857,0.7333,0.7586,0.5923,0.5933
3,0.6620,0.7155,0.4815,0.5652,0.5200,0.2617,0.2637
4,0.8169,0.8721,0.7778,0.7500,0.7636,0.6143,0.6146
5,0.8028,0.9066,0.8519,0.6970,0.7667,0.5989,0.6079
6,0.8732,0.9099,0.7778,0.8750,0.8235,0.7252,0.7282
7,0.7746,0.8426,0.8148,0.6667,0.7333,0.5416,0.5498
8,0.7887,0.8460,0.6667,0.7500,0.7059,0.5419,0.5442


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

最佳參數組合: {'max_depth': 8, 'n_estimators': 362, 'learning_rate': 0.23250825370776484}


In [16]:
# 使用最佳參數訓練模型
optimized_model = create_model(
    'xgboost',
    max_depth=study.best_params['max_depth'],
    n_estimators=study.best_params['n_estimators'],
    learning_rate=study.best_params['learning_rate']
)

# 評估優化後模型的性能
evaluate_model(optimized_model)


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7639,0.7679,0.6429,0.7200,0.6792,0.4934,0.4954
1,0.8611,0.8831,0.8214,0.8214,0.8214,0.7078,0.7078
2,0.7746,0.8738,0.7143,0.7143,0.7143,0.5282,0.5282
3,0.6479,0.7315,0.4815,0.5417,0.5098,0.2366,0.2376
4,0.8169,0.8813,0.7778,0.7500,0.7636,0.6143,0.6146
5,0.8028,0.9175,0.8519,0.6970,0.7667,0.5989,0.6079
6,0.8732,0.9024,0.7778,0.8750,0.8235,0.7252,0.7282
7,0.7746,0.8628,0.8148,0.6667,0.7333,0.5416,0.5498
8,0.7606,0.8409,0.6296,0.7083,0.6667,0.4809,0.4829


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [17]:
# 使用集成方法進行模型融合
blended_model = blend_models([best_model, optimized_model])

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7639,0.7679,0.5714,0.7619,0.6531,0.4796,0.4910
1,0.8611,0.9140,0.7857,0.8462,0.8148,0.7039,0.7052
2,0.8028,0.8895,0.7143,0.7692,0.7407,0.5820,0.5830
3,0.6901,0.7365,0.4074,0.6471,0.5000,0.2919,0.3083
4,0.8169,0.8493,0.7407,0.7692,0.7547,0.6087,0.6090
5,0.8732,0.9141,0.8519,0.8214,0.8364,0.7330,0.7333
6,0.8873,0.8906,0.8148,0.8800,0.8462,0.7575,0.7589
7,0.7887,0.8931,0.8148,0.6875,0.7458,0.5672,0.5732
8,0.7887,0.8266,0.6296,0.7727,0.6939,0.5351,0.5417


Processing:   0%|          | 0/6 [00:00<?, ?it/s]

In [18]:
# 保存模型到本地
save_model(blended_model, 'titanic_optimized_blended_model')


Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('numerical_imputer',
                  TransformerWrapper(exclude=None,
                                     include=['Pclass', 'Age', 'SibSp', 'Parch',
                                              'Fare'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strategy='mean'))),
                 ('categorical_imputer',
                  TransformerWrapper(exclude=None, include=['Sex...
                                                              max_cat_threshold=None,
                                             

In [19]:
from google.colab import files

# 壓縮並下載模型文件
files.download('titanic_optimized_blended_model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>